In [ ]:
import sys, os
sys.path.append(os.path.abspath('../..'))
from status_fields.status_fields_config_oct3v1_brws_250525 import STATUS_FIELDS_CONFIG
from utlis.scan_engine_utlis.scan_eng_big_utlis import log_folder_to_parquet_sep
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files

base_folder = "/data/big_rim/rsync_dcc_sum/Oct3V1" #"/data/big_rim/rsync_dcc_sum/24summ" #"/data/big_rim/rsync_dcc_sum/25Apri_social" #"/data/big_rim/rsync_dcc_sum/Oct3V1" #"/hpc/group/tdunn/Bryan_Rigs/BigOpenField/24summ"  # Replace with your base folder
# save_path = os.path.join(base_folder, 'paret')
failed_paths_file = '/data/big_rim/rsync_dcc_sum/Oct3V1/sync_failed_brws.txt'  # File containing failed paths


force_rescan_rec_files = [
    # ('2023-10-01', '001'),
    # ('2023-10-02', '002'),
    # Add more as needed
]
rescan_threshold_days = 0.0001 # 7 days, but guess if i mess up i can just change it to automatically rescan all, smile... #0.1

log_folder_to_parquet_sep(base_folder, failed_paths_file, STATUS_FIELDS_CONFIG,
                            force_rescan_rec_files=force_rescan_rec_files,
                            rescan_threshold_days=rescan_threshold_days)


all_df = read_all_parquet_files(base_folder)

In [ ]:
import pyarrow.compute as pc
from functools import reduce


table = all_df #combined_df
# Filter mir_generate_param == 0 and sync != 3
conditions = [
   # pc.equal(table['mir_generate_param'], '0'),
   # pc.equal(table['sync'], '1'),
   # pc.not_equal(table['sync'], '3'),
   # pc.equal(table['com'], '1'),
   # # pc.equal(table['com_vis'], '1'),
   # # pc.equal(table['v1'], '1'),
   # pc.equal(table['dannce'], '1'),
   #  
   # pc.equal(table['test'], '1'),
   # pc.equal(table['dannce_vis'], '1'),
   pc.equal(table['social'], '1'),  
   pc.equal(table['mini_6cam_map'], '1'),
   # pc.equal(table['date_folder'], '2025_05_16'),
   pc.equal(table['mini_rec_sync_com'], '1'),
   # pc.equal(table['mini_rec_sync'], '1'),
   #mini_rec_sync mini_rec_sync_com
   # mini_6cam_map
]

filter_mask = reduce(pc.and_, conditions)


# Apply the filter and print the results
filtered_table = table.filter(filter_mask)

# Print each row of the filtered table
print(filtered_table.to_pandas())  # This will display the filtered data in a familiar pandas-like format


In [ ]:
# Convert and flatten the ChunkedArray
rec_paths = filtered_table["rec_path"].to_pylist()

rec_paths

# export

In [ ]:
from export_profiles import load_profiles, build_export_plan, export_with_rsync

profiles = load_profiles("profiles.yaml")
profile  = profiles["paper-core"]   # pick one: paper-core | qc-review | dannce-delta | repro-full


In [ ]:
# pairs_manifest.py
from pathlib import Path
from typing import Optional, Dict, Tuple
import re
import pyarrow as pa, pyarrow.compute as pc
import pyarrow.csv as pacsv

PAIR_COLS = [
    "source_rec_path","animal1_key","animal2_key",
    "area1","area2","short_id1","short_id2",
    "session_manual","include","notes"
]

# ----- utilities -----
def _find_source_col(t: pa.Table, source_col: Optional[str]) -> str:
    if source_col and source_col in t.schema.names:
        return source_col
    for c in ["source_rec_path","source_path","rec_path","recording_path","root_time_path","session_path","path"]:
        if c in t.schema.names:
            return c
    raise ValueError(f"Need a path column; existing columns: {list(t.schema.names)}")

# def _truthy(val) -> bool:
#     s = str(val).strip().lower()
#     return s in {"", "1", "y", "yes", "true"}  # blank means include

def _truthy(val) -> bool:
    s = str(val).strip().lower()
    # Only treat these as 'exclude'. Everything else (incl. blank/none/unk) = include.
    return s not in {"0", "no", "n", "false", "skip", "exclude"}

# def _parse_short_id(s: str) -> Tuple[str, int]:
#     m = re.match(r"^([A-Za-z]+)\s*(\d+)$", s.strip())
#     return (m.group(1).upper(), int(m.group(2))) if m else ("", 0)

# ----- 1) seed/append pairs CSV from a scan (no guessing) -----
def seed_pairs_csv(
    filtered_table: pa.Table,
    out_csv: str,
    prev_csv: Optional[str] = None,
    source_col: Optional[str] = None,
) -> str:
    t = filtered_table
    if "is_social" in t.schema.names:
        t = t.filter(pc.field("is_social") == True)

    src_col = _find_source_col(t, source_col)

    # unique non-null string paths
    arr = t[src_col]
    if isinstance(arr, pa.ChunkedArray):
        arr = arr.combine_chunks()
    arr = pc.cast(arr, pa.string())
    arr = pc.filter(arr, pc.invert(pc.is_null(arr)))
    src = pc.unique(arr)

    blanks = {c: pa.array([""] * len(src), type=pa.string()) for c in PAIR_COLS[1:]}
    fresh = pa.concat_tables([pa.table({"source_rec_path": src}), pa.table(blanks)], promote=True)

    if prev_csv and Path(prev_csv).exists():
        prev = pacsv.read_csv(prev_csv)
        is_dup = pc.is_in(fresh["source_rec_path"], value_set=prev["source_rec_path"])
        out = pa.concat_tables([prev, fresh.filter(pc.invert(is_dup))], promote=True)
    else:
        out = fresh

    pacsv.write_csv(out, out_csv)
    return out_csv

# ----- 2) optional: auto-fill stable short IDs inside the same CSV -----
# Rule: same animal_key -> same short_id forever (persisted in this CSV).
# It DOES NOT infer area; if area1/area2 are blank for a key, that key is skipped.
# It never changes existing short_ids, only fills missing ones.
# --- put these helpers near the top of pairs_manifest.py ---
import re

_PLACEHOLDER_EMPTY = {"", "none", "null", "na", "n/a", "nan"}

def _norm_str(x) -> str:
    return str(x).strip() if x is not None else ""

def _norm_blank(x) -> str:
    s = _norm_str(x)
    return "" if s.lower() in _PLACEHOLDER_EMPTY else s

def _norm_area(x) -> str:
    s = _norm_blank(x).upper()
    if s in {"MC", "VC"}:
        return s
    # treat UNK / UNKNOWN / ? as blank
    if s in {"UNK", "UNKNOWN", "?"}:
        return ""
    return ""  # anything else -> blank

def _parse_sid(x):
    """
    Returns (prefix, num). Bare 'MC'/'VC' and placeholders count as empty.
    Accepts 'mc12', 'VC 7', etc. Canonical prefix is uppercased.
    """
    s = _norm_blank(x).upper()
    if s in {"MC", "VC", ""}:
        return ("", 0)
    m = re.match(r"^(MC|VC)\s*(\d+)$", s)
    if not m:
        return ("", 0)
    return (m.group(1), int(m.group(2)))

def _canon_sid(prefix: str, n: int) -> str:
    return f"{prefix}{n}" if prefix and n > 0 else ""
# --- end helpers ---


# Simple: canonical_key -> (area, short_id) map from a separate registry
def autofill_short_ids_in_pairs(pairs_csv: str) -> str:
    t = pacsv.read_csv(pairs_csv)
    rows = t.to_pylist()
    
    # Single source of truth per animal
    animals = {}  # animal_key -> {"area": str, "num": int}
    used = {"MC": set(), "VC": set(), "UNK": set()}
    
    # First pass: collect existing data
    for r in rows:
        for animal_key, area_val, sid_val in [
            (r.get("animal1_key"), r.get("area1"), r.get("short_id1")),
            (r.get("animal2_key"), r.get("area2"), r.get("short_id2"))
        ]:
            animal = _norm_blank(animal_key)
            if not animal:
                continue
                
            area = _norm_area(area_val)
            prefix, num = _parse_sid(sid_val)
            
            if animal not in animals:
                animals[animal] = {"area": area or prefix or "", "num": num}
            else:
                # Update area if we have a better one
                if area and not animals[animal]["area"]:
                    animals[animal]["area"] = area
                # Update num if we found an actual ID
                if num > 0:
                    animals[animal]["num"] = num
            
            if num > 0:
                bucket = prefix or area or "UNK"
                used.setdefault(bucket, set()).add(num)
    
    # Assign missing IDs
    for animal, info in animals.items():
        if info["num"] == 0:
            bucket = info["area"] if info["area"] in {"MC", "VC"} else "UNK"
            n = 1
            while n in used[bucket]:
                n += 1
            used[bucket].add(n)
            info["num"] = n
            if not info["area"]:
                info["area"] = bucket
    
    # Write back
    for r in rows:
        for side in [("animal1_key", "area1", "short_id1"), 
                     ("animal2_key", "area2", "short_id2")]:
            animal = _norm_blank(r.get(side[0]))
            if animal and animal in animals:
                r[side[1]] = animals[animal]["area"]
                r[side[2]] = _canon_sid(animals[animal]["area"], animals[animal]["num"])
    
    out_tab = pa.table({c: pa.array([row.get(c, "") for row in rows], type=pa.string()) for c in PAIR_COLS})
    pacsv.write_csv(out_tab, pairs_csv)
    return pairs_csv


# ----- 3) build a deterministic plan from social_pairs.csv only -----

def build_plan_from_pairs(pairs_csv: str, profile: dict) -> list[dict]:
    t = pacsv.read_csv(pairs_csv)
    rows = t.to_pylist()

    # keep rows with both animals non-empty, and not explicitly excluded
    keep = []
    for r in rows:
        a1 = str(r.get("animal1_key", "")).strip()
        a2 = str(r.get("animal2_key", "")).strip()
        inc = _truthy(r.get("include", ""))
        if a1 and a2 and inc:
            keep.append(r)

    if not keep:
        return []  # nothing to export

    includes = profile.get("includes") or profile.get("path_globs") or []

    # group by pair label; prefer short IDs if present
    from collections import defaultdict
    groups = defaultdict(list)
    for r in keep:
        lab1 = (str(r.get("short_id1","")).strip() or str(r.get("animal1_key","")).strip())
        lab2 = (str(r.get("short_id2","")).strip() or str(r.get("animal2_key","")).strip())
        pair_key = "+".join(sorted([lab1, lab2]))
        r["_pair_key"] = pair_key
        groups[pair_key].append(r)

    plan = []
    for pair_key, items in groups.items():
        items.sort(key=lambda r: str(r.get("source_rec_path","")))
        for i, r in enumerate(items, 1):
            s_manual = str(r.get("session_manual","")).strip()
            try:
                sidx = int(s_manual) if s_manual else i
            except ValueError:
                sidx = i
            plan.append({
                "dest_rel_path": f"social/{pair_key}_s{sidx}/",
                "source_rec_path": str(r.get("source_rec_path","")),
                "id": pair_key,
                "session_index": sidx,
                "path_globs": includes,
            })
    return plan


In [ ]:
seed_pairs_csv(
    filtered_table,
    out_csv="./social_pairs.csv",
    prev_csv="./social_pairs.csv",
    source_col="rec_path",
)
autofill_short_ids_in_pairs("./social_pairs.csv")
plan = build_plan_from_pairs("./social_pairs.csv", profile=profile)